In [1]:
import numpy as np
import tensorflow as tf

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
num_classes = 3
initial_lr = 5e-4
weight_decay = 1e-5
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [4]:
from network import build_vit
from train import WarmUpCosine, AdamW, LastNSaver, make_train_ds, make_test_ds, RSquared 

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [6]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()
y_train = tf.cast(y_train, dtype=tf.float32)
y_test = tf.cast(y_test, dtype=tf.float32)

✔ 检测到缓存数据，已直接加载


In [7]:
model = build_vit(num_classes=1)

In [8]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay
)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RSquared()]
)

In [9]:
saver = LastNSaver(n=20)

In [14]:
train_ds = make_train_ds(
    x_train, y_train,
    batch_size=batch_size,
    image_size=32, pad=4,
    mixup_alpha=0.05   # 0.2~0.4 common; increase if overfitting
)
test_ds = make_test_ds(x_test, y_test, batch_size=batch_size)

model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
157/157 - 22s - loss: 0.0314 - r_squared: -5.0256e-02 - val_loss: 0.0340 - val_r_squared: -3.9770e-02 - 22s/epoch - 143ms/step
Epoch 2/200
157/157 - 12s - loss: 0.0283 - r_squared: 0.0590 - val_loss: 0.0331 - val_r_squared: -1.3512e-02 - 12s/epoch - 74ms/step
Epoch 3/200
157/157 - 12s - loss: 0.0261 - r_squared: 0.1255 - val_loss: 0.0284 - val_r_squared: 0.1297 - 12s/epoch - 74ms/step
Epoch 4/200
157/157 - 12s - loss: 0.0256 - r_squared: 0.1501 - val_loss: 0.0342 - val_r_squared: -4.7929e-02 - 12s/epoch - 74ms/step
Epoch 5/200
157/157 - 12s - loss: 0.0247 - r_squared: 0.1882 - val_loss: 0.0242 - val_r_squared: 0.2608 - 12s/epoch - 74ms/step
Epoch 6/200
157/157 - 12s - loss: 0.0219 - r_squared: 0.2687 - val_loss: 0.0227 - val_r_squared: 0.3050 - 12s/epoch - 74ms/step
Epoch 7/200
157/157 - 12s - loss: 0.0205 - r_squared: 0.3184 - val_loss: 0.0311 - val_r_squared: 0.0482 - 12s/epoch - 74ms/step
Epoch 8/200
157/157 - 12s - loss: 0.0200 - r_squared: 0.3451 - val_loss: 0.0180 - v

In [15]:
model.save("ViT_8.h5")